# 03 - Variance Decomposition in the Dynamic Factor Model

## Overview

A key output of the DFM is the **variance decomposition**: how much of
each observed series' variation is explained by the common factors vs.
idiosyncratic noise.

### Decomposition

For series $i$ with loading $\lambda_i$ on factor $f_t$:

$$
\text{Var}(y_{it}) = \underbrace{\lambda_i^2 \cdot \text{Var}(f_t)}_{\text{common}} + \underbrace{\sigma_i^2}_{\text{idiosyncratic}}
$$

The **R-squared** for series $i$ is:

$$
R^2_i = \frac{\sum_j \lambda_{ij}^2}{\sum_j \lambda_{ij}^2 + \sigma_i^2}
$$

This measures the fraction of variance explained by the common factors.
With $K$ factors, we can also decompose the contribution of each factor
individually.

### This Notebook

We fit a DFM with $K=2$ factors to the `us_macro_panel.csv` and compute:
1. $R^2$ for each series
2. Per-factor contribution
3. Rolling variance decomposition over time
4. Impulse responses and historical decomposition
5. Comparison with VAR-based FEVD (statsmodels)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from kalmanbox import DynamicFactorModel

import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 100,
})

print('Imports OK')

In [ ]:
# Load data and fit DFM with K=2 factors
data_dir = Path(__file__).resolve().parent / 'data' if '__file__' in dir() else Path('data')
df = pd.read_csv(data_dir / 'us_macro_panel.csv', parse_dates=['date'])
df = df.set_index('date')

series_names = df.columns.tolist()
y = df.values.astype(np.float64)
dates = df.index

model = DynamicFactorModel(y, k_factors=2, factor_order=1,
                            endog_names=series_names)
results = model.fit(compute_se=False)

print(f'Model fitted: K=2 factors, {len(series_names)} series')
print(f'Log-Likelihood: {results.loglike:.2f}')
print(f'BIC: {results.bic:.2f}')

## R-squared by Series

We compute $R^2_i$ for each series using kalmanbox's built-in
`variance_decomposition()` method. This returns a matrix of shape
$(N, K+1)$ where each row sums to 1.

In [ ]:
# Variance decomposition using kalmanbox
decomp = model.variance_decomposition(results)
# decomp shape: (N, K+1) = (15, 3) -> [Factor1, Factor2, Idiosyncratic]

decomp_df = pd.DataFrame(
    decomp,
    index=series_names,
    columns=['Factor 1', 'Factor 2', 'Idiosyncratic']
)

# R^2 = 1 - Idiosyncratic = Factor1 + Factor2
decomp_df['R_squared'] = 1 - decomp_df['Idiosyncratic']

print('Variance Decomposition (proportions):')
print(decomp_df.round(4).to_string())
print(f'\nRow sums (should all be 1.0): {decomp.sum(axis=1).round(6)}')

In [ ]:
# Heatmap of R^2 by series
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# R^2 bar chart sorted
r2_sorted = decomp_df['R_squared'].sort_values(ascending=True)
colors = plt.cm.RdYlGn(r2_sorted.values)
axes[0].barh(r2_sorted.index, r2_sorted.values, color=colors, edgecolor='gray')
axes[0].set_xlabel('$R^2$ (fraction explained by factors)')
axes[0].set_title('$R^2$ by Series (K=2 DFM)')
axes[0].set_xlim(0, 1)
for i, (name, val) in enumerate(r2_sorted.items()):
    axes[0].text(val + 0.02, i, f'{val:.2%}', va='center', fontsize=9)

# Stacked bar: Factor 1, Factor 2, Idiosyncratic
decomp_plot = decomp_df[['Factor 1', 'Factor 2', 'Idiosyncratic']].loc[r2_sorted.index]
decomp_plot.plot(kind='barh', stacked=True, ax=axes[1],
                 color=['#3498db', '#e74c3c', '#95a5a6'], edgecolor='white')
axes[1].set_xlabel('Variance Share')
axes[1].set_title('Variance Decomposition: Factor 1 + Factor 2 + Idiosyncratic')
axes[1].legend(loc='lower right')
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()

## Per-Factor Contribution by Series

We examine which factor drives which series most, helping with economic
interpretation of the latent factors.

In [ ]:
# Contribution heatmap
fig, ax = plt.subplots(figsize=(8, 10))
contrib = decomp_df[['Factor 1', 'Factor 2']].copy()
# Normalize to show relative contribution of each factor (among common factors)
contrib_rel = contrib.div(contrib.sum(axis=1), axis=0)

sns.heatmap(contrib_rel, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Relative Contribution'})
ax.set_title('Relative Factor Contribution (among common factors)')
ax.set_ylabel('Series')
plt.tight_layout()
plt.show()

# Identify which series load more on each factor
print('Series primarily driven by Factor 1:')
f1_dominant = contrib_rel[contrib_rel['Factor 1'] > 0.5].index
for s in f1_dominant:
    print(f'  {s}: Factor 1 = {contrib_rel.loc[s, "Factor 1"]:.1%}')

print(f'\nSeries primarily driven by Factor 2:')
f2_dominant = contrib_rel[contrib_rel['Factor 2'] > 0.5].index
for s in f2_dominant:
    print(f'  {s}: Factor 2 = {contrib_rel.loc[s, "Factor 2"]:.1%}')

## Variance Explained Over Time (Rolling Window)

We compute the variance decomposition using a rolling window to see how
the factor structure evolves over time. This can reveal structural changes
in the economy.

In [ ]:
# Rolling variance decomposition
# For each rolling window, compute the empirical R^2 using the smoothed factors
factors = results.smoothed_state[:, :2]  # (T, 2)
Lambda = results.ssm.Z[:, :2]           # (N, 2)
R_diag = np.diag(results.ssm.H)         # (N,)

window = 60  # 5-year rolling window
T = len(dates)
N = len(series_names)

# Compute rolling R^2 for each series
rolling_r2 = np.full((T, N), np.nan)

for t in range(window, T):
    y_win = y[t - window:t, :]
    f_win = factors[t - window:t, :]

    for i in range(N):
        # Fitted values from factors
        y_hat_i = f_win @ Lambda[i, :]
        var_total = np.var(y_win[:, i])
        var_explained = np.var(y_hat_i)
        if var_total > 0:
            rolling_r2[t, i] = var_explained / var_total

# Plot rolling average R^2 across all series
avg_r2 = np.nanmean(rolling_r2, axis=1)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Panel 1: Average R^2 over time
ax = axes[0]
ax.plot(dates[window:], avg_r2[window:], 'b-', linewidth=1.5)
ax.set_ylabel('Average $R^2$')
ax.set_title(f'Rolling Average $R^2$ Across All Series (window={window} months)')
ax.set_ylim(0, 1)
ax.axhline(np.nanmean(avg_r2[window:]), color='red', linestyle='--', alpha=0.5,
           label=f'Mean = {np.nanmean(avg_r2[window:]):.2%}')
ax.legend()

# Panel 2: R^2 for selected series
ax = axes[1]
selected = ['gdp_growth', 'industrial_production', 'cpi_inflation', 'sp500_returns']
for s in selected:
    idx = series_names.index(s)
    ax.plot(dates[window:], rolling_r2[window:, idx], linewidth=1.2, label=s)
ax.set_ylabel('$R^2$')
ax.set_title(f'Rolling $R^2$ for Selected Series (window={window} months)')
ax.set_ylim(0, 1)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Impulse Response: Shock to Factor

We compute the impulse response of each observed series to a unit shock
in each factor. The impulse response at horizon $h$ is:

$$
\text{IRF}_i(h) = \lambda_i \cdot \Phi^h
$$

This shows how a shock to the latent factor propagates to the observed
series over time.

In [ ]:
# Impulse response functions
Phi = results.ssm.T[:2, :2]  # Factor transition (2x2)
n_horizons = 24

# IRF: response of each series to a unit shock in each factor
irf = np.zeros((n_horizons, N, 2))  # (horizon, series, shock_factor)

for h in range(n_horizons):
    Phi_h = np.linalg.matrix_power(Phi, h)
    for shock_j in range(2):
        # Unit shock to factor j
        e_j = np.zeros(2)
        e_j[shock_j] = 1.0
        factor_response = Phi_h @ e_j  # (2,)
        for i in range(N):
            irf[h, i, shock_j] = Lambda[i, :] @ factor_response

# Plot IRF for selected series
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
selected_series = ['gdp_growth', 'industrial_production', 'unemployment',
                   'cpi_inflation', 'sp500_returns', 'fed_funds_rate']

for shock_j in range(2):
    ax = axes[shock_j]
    for s in selected_series:
        idx = series_names.index(s)
        ax.plot(range(n_horizons), irf[:, idx, shock_j], linewidth=1.5, label=s)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Horizon (months)')
    ax.set_ylabel('Response')
    ax.set_title(f'Impulse Response to Factor {shock_j + 1} Shock')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## Historical Decomposition

The historical decomposition shows the contribution of each factor to
each observed series at every point in time:

$$
y_{it} = \lambda_{i1} \cdot f_{1t} + \lambda_{i2} \cdot f_{2t} + \varepsilon_{it}
$$

We use the smoothed factor estimates to separate each component.

In [ ]:
# Historical decomposition for selected series
factors = results.smoothed_state[:, :2]

# Contribution of each factor to each series at each time point
contrib_f1 = factors[:, 0:1] * Lambda[:, 0:1].T  # (T, N)
contrib_f2 = factors[:, 1:2] * Lambda[:, 1:2].T  # (T, N)
idiosyncratic = y - contrib_f1 - contrib_f2       # (T, N)

# Plot for 4 selected series
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)
selected = ['gdp_growth', 'industrial_production', 'cpi_inflation', 'sp500_returns']

for ax, s in zip(axes, selected):
    idx = series_names.index(s)
    ax.fill_between(dates, 0, contrib_f1[:, idx], alpha=0.4, color='#3498db',
                    label='Factor 1')
    ax.fill_between(dates, contrib_f1[:, idx],
                    contrib_f1[:, idx] + contrib_f2[:, idx],
                    alpha=0.4, color='#e74c3c', label='Factor 2')
    ax.plot(dates, y[:, idx], 'k-', linewidth=0.8, alpha=0.7, label='Observed')
    ax.set_ylabel(s)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.3)
    if ax == axes[0]:
        ax.legend(loc='upper right', fontsize=8)

axes[0].set_title('Historical Decomposition: Factor Contributions to Each Series')
axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

## Comparison with VAR FEVD (statsmodels)

We compare the DFM variance decomposition with the **Forecast Error Variance
Decomposition** (FEVD) from a VAR model on the same data. The FEVD from
a reduced-form VAR is conceptually different (it decomposes forecast error
variance across VAR innovations), but provides a useful reference point.

In [ ]:
# VAR FEVD comparison using statsmodels
from statsmodels.tsa.api import VAR

# Fit VAR on a subset of series (VAR with 15 series is overparameterized)
var_series = ['gdp_growth', 'industrial_production', 'unemployment',
              'cpi_inflation', 'sp500_returns']
y_var = df[var_series].values

var_model = VAR(y_var)
var_results = var_model.fit(maxlags=2, ic='bic')
fevd = var_results.fevd(12)

# Plot FEVD for GDP at horizon 12
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# DFM decomposition for the same series
ax = axes[0]
dfm_r2 = []
for s in var_series:
    idx = series_names.index(s)
    dfm_r2.append(decomp_df.loc[s, ['Factor 1', 'Factor 2', 'Idiosyncratic']].values)
dfm_r2 = np.array(dfm_r2)

x_pos = np.arange(len(var_series))
bars1 = ax.bar(x_pos, dfm_r2[:, 0], color='#3498db', label='Factor 1')
bars2 = ax.bar(x_pos, dfm_r2[:, 1], bottom=dfm_r2[:, 0], color='#e74c3c', label='Factor 2')
bars3 = ax.bar(x_pos, dfm_r2[:, 2], bottom=dfm_r2[:, 0] + dfm_r2[:, 1],
               color='#95a5a6', label='Idiosyncratic')
ax.set_xticks(x_pos)
ax.set_xticklabels(var_series, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Variance Share')
ax.set_title('DFM Variance Decomposition')
ax.legend(fontsize=8)
ax.set_ylim(0, 1)

# VAR FEVD at horizon 12
# fevd.decomp shape: (neqs, steps, neqs)
ax = axes[1]
fevd_h12 = fevd.decomp[:, 11, :]  # horizon 12 for all equations
bottom = np.zeros(len(var_series))
colors = plt.cm.Set2(np.linspace(0, 1, len(var_series)))
for j in range(len(var_series)):
    ax.bar(x_pos, fevd_h12[:, j], bottom=bottom, color=colors[j],
           label=var_series[j])
    bottom += fevd_h12[:, j]
ax.set_xticks(x_pos)
ax.set_xticklabels(var_series, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Variance Share')
ax.set_title('VAR FEVD (horizon=12 months)')
ax.legend(fontsize=7, loc='upper right')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print('Note: DFM decomposes variance into common factors + idiosyncratic noise,')
print('while VAR FEVD decomposes forecast error variance across all variables.')
print('The two are conceptually different but both reveal cross-variable dependencies.')

## Conclusions

1. **R-squared by series**: The DFM's variance decomposition reveals which
   series are well-explained by the common factors (high $R^2$) vs. driven
   mainly by idiosyncratic noise (low $R^2$).

2. **Per-factor contribution**: With $K=2$ factors, we can identify which
   series are primarily driven by Factor 1 vs Factor 2, aiding economic
   interpretation.

3. **Rolling variance explained**: The rolling-window $R^2$ shows how the
   factor structure evolves over time. Higher $R^2$ during recessions
   indicates that the common factor (business cycle) becomes more dominant.

4. **Impulse responses**: A unit shock to each factor propagates to the
   observed series through the loadings $\Lambda$ and decays at rate $\Phi$.

5. **Historical decomposition**: At each time point, the observed value of
   each series can be decomposed into contributions from Factor 1, Factor 2,
   and the idiosyncratic component.

6. **DFM vs VAR FEVD**: The DFM decomposes variance into a small number of
   latent factors, while the VAR FEVD decomposes forecast error across all
   observed variables. Both are useful but answer different questions.

### References

- Stock, J.H. and Watson, M.W. (2002). "Forecasting Using Principal Components
  from a Large Number of Predictors."
- Bai, J. and Ng, S. (2002). "Determining the Number of Factors in Approximate
  Factor Models."